LangGraph Conversational Agent with Memory and Safety Guardrails.

This module implements a stateful, multi-thread chatbot using LangGraph and OpenAI, 
designed for safe, persistent, and interactive user experiences.

Core Architecture:
    * Input Moderation: A custom Pydantic-driven guardrail node evaluates all 
      user inputs for safety prior to standard LLM processing.
    * Persistent Memory: SQLite-backed check-pointing allows the system to 
      save, resume, and cleanly switch between multiple chat threads.
    * Dynamic Routing: Conditional graph edges automatically intercept safe/unsafe 
      prompts and generate context-aware titles for new conversations.

In [ ]:
import sqlite3
import uuid
import gradio as gr
from typing import Annotated
from typing_extensions import TypedDict
from dotenv import load_dotenv
from IPython.display import Image, display
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage


In [ ]:
load_dotenv(override=True)

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages] # Tracks the conversation history
    is_safe: bool                           # True if input passed the safety guardrail
    title: str                              # Short summary generated for the UI sidebar

In [ ]:
class InputGuardrail(BaseModel):
    is_safe: bool = Field(description="True if the user input is safe. False if it asks for malicious code, system commands, or inappropriate content.")

In [ ]:
# Create the Title Generation Node
def generate_title_node(state: State):
    """Generates a short title based on the first user message."""
    # The first message [0] is the SystemMessage. The first HumanMessage is usually [1], but we will search to be safe.
    first_user_message = [m.content for m in state["messages"] if isinstance(m, HumanMessage)][0]
    
    prompt = f"Summarize this user query in 3 to 4 words to use as a chat title. Do not use quotes. Query: {first_user_message}"
    
    title_response = llm.invoke(prompt)
    
    return {"title": title_response.content}

In [ ]:
# Create a router to check if we already have a title
def route_after_chatbot(state: State):
    """If the title is missing, generate one. Otherwise, we are done."""
    if not state.get("title"):
        return "generate_title"
    return END

In [ ]:
def guardrail_node(state: State):
    """Evaluates the user's latest message for safety."""
    user_message = state["messages"][-1].content
    
    analyzer = llm.with_structured_output(InputGuardrail)
    
    # input validation prompt for the guardrail
    prompt = f"""
    You are a security guardrail. Analyze the following user input. 
    Is it safe and appropriate? 
    Input: {user_message}
    """

    result = analyzer.invoke(prompt)
    
    # Update the graph's state with the boolean result
    return {"is_safe": result.is_safe}

In [ ]:
def refusal_node(state: State):
    """Generates a standard rejection message."""
    rejection = AIMessage(content="I cannot fulfill this request as it violates my safety guidelines.")
    return {"messages": [rejection]}

In [ ]:
def route_after_guardrail(state: State):
    """Decides the next node based on the is_safe flag."""
    if state.get("is_safe") == True:
        return "chatbot"
    else:
        return "refusal"

In [ ]:
def chatbot_node(state: State):
    messages = state["messages"]
    
    if not isinstance(messages[0], SystemMessage):
        system_prompt = SystemMessage(
            content="You are a helpful AI assistant. You must never write code that executes arbitrary system commands. If asked about harmful topics, politely decline."
        )
        messages = [system_prompt] + messages
        
    response = llm.invoke(messages)
    return {"messages": [response]}

In [ ]:
# Initialize the graph
graph_builder = StateGraph(State)

# Add all four nodes
graph_builder.add_node("guardrail", guardrail_node)
graph_builder.add_node("chatbot", chatbot_node)
graph_builder.add_node("refusal", refusal_node)
graph_builder.add_node("generate_title", generate_title_node)

# START always goes to the guardrail first
graph_builder.add_edge(START, "guardrail")

# The conditional edge decides where to go after the guardrail
graph_builder.add_conditional_edges(
    "guardrail", 
    route_after_guardrail,
    {"chatbot": "chatbot", "refusal": "refusal"}
)

# The refusal node ends the cycle immediately
graph_builder.add_edge("refusal", END)

# The chatbot conditionally routes to generate a title, or ends the cycle
graph_builder.add_conditional_edges(
    "chatbot", 
    route_after_chatbot,
    {"generate_title": "generate_title", END: END}
)

# The title generator leads to the END of the cycle
graph_builder.add_edge("generate_title", END)



In [ ]:
# Set up SQLite Memory to save our threads
conn = sqlite3.connect("my_chat_history.db", check_same_thread=False)
memory = SqliteSaver(conn)

In [ ]:
graph = graph_builder.compile(checkpointer=memory)

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# Maintain an ordered list of Thread IDs and a mapping to their display titles
thread_ids = [str(uuid.uuid4())]
thread_titles = {thread_ids[0]: "New Chat"}

In [ ]:
# Helper Functions for Gradio

def get_gradio_history(thread_id: str):
    """Fetches the history from LangGraph and converts it for Gradio."""
    config = {"configurable": {"thread_id": thread_id}}
    state = graph.get_state(config)
    
    # If the thread is new and has no state yet, return an empty list
    if not state or "messages" not in state.values:
        return []
    
    gradio_messages = []
    for msg in state.values["messages"]:
        role = "user" if isinstance(msg, HumanMessage) else "assistant"
        gradio_messages.append({"role": role, "content": msg.content})
        
    return gradio_messages



In [ ]:
def get_chat_list():
    """Generates the 2D array for the Dataframe based on our ordered list."""
    return [[thread_titles[tid]] for tid in thread_ids]

In [ ]:
def start_new_chat():
    """Creates a new thread, adds it to the list, and sets it as active."""
    new_tid = str(uuid.uuid4())
    thread_ids.append(new_tid)
    thread_titles[new_tid] = "New Chat"
    
    # Returns: (Updated Dataframe, Cleared Chatbot, New Active Thread ID)
    return get_chat_list(), [], new_tid

In [ ]:
def switch_chat(evt: gr.SelectData):
    """Uses the row index of the clicked cell to load the correct history."""
    row_index = evt.index[0] # Gets the row number (0, 1, 2, etc.)
    selected_tid = thread_ids[row_index]
    
    # Returns: (Updated Chatbot, Selected Thread ID)
    return get_gradio_history(selected_tid), selected_tid

In [ ]:
def submit_message(user_input, current_thread_id):
    """Sends the message, retrieves the new title, and updates the UI."""
    if not user_input.strip():
        return gr.update(), gr.update(), gr.update() 

    config = {"configurable": {"thread_id": current_thread_id}}
    
    # 1. Run the graph
    graph.invoke(
        {"messages": [{"role": "user", "content": user_input}]}, 
        config=config
    )
    
    # 2. Check the state to see if a new title was generated
    state = graph.get_state(config)
    new_title = state.values.get("title", "New Chat")
    
    # 3. Update our dictionary with the new title
    thread_titles[current_thread_id] = new_title
    
    # 4. Refresh everything
    updated_history = get_gradio_history(current_thread_id)
    
    # Returns: (Updated Chatbot, Cleared Input, Updated Dataframe Sidebar)
    return updated_history, "", get_chat_list()

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    # State holds the actual UUID of the active thread
    current_thread = gr.State(thread_ids[0])
    
    gr.Markdown("## My Chat Assistant with History")
    
    with gr.Row():
        with gr.Column(scale=1):
            new_chat_btn = gr.Button("➕ New Chat", variant="primary")
            chat_list_ui = gr.Dataframe(
                headers=["Your Conversations"],
                value=get_chat_list(),
                interactive=False,
                row_count=(1, "dynamic"),
                col_count=(1, "fixed")
            )
            
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=500, type="messages")
            msg_input = gr.Textbox(placeholder="Type your message here and press Enter...", show_label=False)
            
    # --- Event Listeners ---
    
    # Submitting a message updates the Chatbot, Input box, AND the Dataframe titles
    msg_input.submit(
        submit_message, 
        inputs=[msg_input, current_thread], 
        outputs=[chatbot, msg_input, chat_list_ui]
    )
    
    new_chat_btn.click(
        start_new_chat, 
        inputs=[], 
        outputs=[chat_list_ui, chatbot, current_thread]
    )
    
    chat_list_ui.select(
        switch_chat, 
        inputs=[], 
        outputs=[chatbot, current_thread]
    )

In [ ]:
demo.launch()